<a href="https://colab.research.google.com/github/tqd3pz/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-25%20%E2%80%94%20Cleaning%20Gauntlet%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [34]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [35]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [36]:
initial_rows = df.shape[0]
df.drop_duplicates(inplace=True)
rows_affected = initial_rows - df.shape[0]
log(step='TODO 1', decision='Dropped duplicate rows', rows_affected=rows_affected)

[TODO 1] Dropped duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [37]:
initial_rows = df.shape[0]
df['price'] = df['price'].astype(str).str.replace('$', '', regex=False)
df['price'] = pd.to_numeric(df['price'], errors='coerce')
# For rows affected, we'll consider rows where price became NaN after conversion
rows_affected = df['price'].isna().sum()
log(step='TODO 2', decision='Cleaned price column to float, handling non-numeric values', rows_affected=rows_affected)

[TODO 2] Cleaned price column to float, handling non-numeric values (0 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [38]:
initial_rows = df.shape[0]
# Convert qty to numeric, coercing errors to NaN
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')

# Count rows that will be dropped (NaN or negative)
rows_to_drop_nan = df['qty'].isna().sum()
rows_to_drop_negative = (df['qty'] < 1).sum()
rows_affected = int(rows_to_drop_nan + rows_to_drop_negative) # Quantities must be >= 1

# Drop rows with NaN in 'qty' or negative 'qty' by creating a new DataFrame
df = df.dropna(subset=['qty']).copy()
df = df[df['qty'] >= 1].copy()

log(step='TODO 3', decision='Cleaned qty column, dropped rows with missing/negative qty', rows_affected=rows_affected)

[TODO 3] Cleaned qty column, dropped rows with missing/negative qty (25 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [39]:
print(df['item'].value_counts())

ITEM_MAP = {
    'cheese burger': 'Cheeseburger',
    'foam finger': 'Foam Finger',
    'rain poncho': 'Rain Poncho'
}

initial_unique_items = df['item'].nunique()
df['item'] = df['item'].replace(ITEM_MAP)
rows_affected = initial_unique_items - df['item'].nunique()

log(step='TODO 4', decision='Canonicalized item names', rows_affected=rows_affected)

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[TODO 4] Canonicalized item names (3 row(s))


In [40]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse

### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [41]:
print(df['category'].value_counts())

CATEGORY_MAP = {
    'food': 'Food',
    'rain-gear': 'RainGear',
    'Apparel': 'Merch' # Business decision to group Apparel under Merch
}

initial_unique_categories = df['category'].nunique()
df['category'] = df['category'].replace(CATEGORY_MAP)
rows_affected = initial_unique_categories - df['category'].nunique()

log(step='TODO 5', decision='Normalized category names', rows_affected=rows_affected)

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[TODO 5] Normalized category names (3 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [42]:
assert df.duplicated().sum() == 0, "Duplicates found!"
assert df['qty'].min() >= 1, "Quantities less than 1 found!"
assert df['price'].dtype == float, "Price column is not float type!"

# Assert that item names are canonical
expected_items = ['Cheeseburger', 'Foam Finger', 'Rain Poncho']
assert all(item in expected_items for item in df['item'].unique()), "Non-canonical item names found!"

# Assert that category names are canonical
expected_categories = ['Food', 'Merch', 'RainGear']
assert all(category in expected_categories for category in df['category'].unique()), "Non-canonical category names found!"

print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [43]:
# Add a revenue column
df['revenue'] = (df['qty'] * df['price']).round(2)

# Print revenue by category, highest first
revenue_by_category = df.groupby('category')['revenue'].sum().sort_values(ascending=False).round(2)
print("Revenue by Category:\n", revenue_by_category)

# Print overall total revenue
overall_total_revenue = df['revenue'].sum().round(2)
print("\nOverall Total Revenue: $", overall_total_revenue)

Revenue by Category:
 category
Food        1656.0
Merch       1572.0
RainGear    1512.0
Name: revenue, dtype: float64

Overall Total Revenue: $ 4740.0


**What I would tell the vendor:** _Based on revenue, the vendor should stock more Food items._

### TODO 8 — read back your log

In [44]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,TODO 1,Dropped duplicate rows,15
1,TODO 2,"Cleaned price column to float, handling non-nu...",0
2,TODO 3,"Cleaned qty column, dropped rows with missing/...",25
3,TODO 4,Canonicalized item names,3
4,TODO 5,Normalized category names,3


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

a) Dropping duplicates changed the revenue total the most. Revenue went from $4,852.50 before removing duplicates to $4,594.50 after.
b) I chose to group Apparel with Merch. Another reasonable choice would have been to keep Apparel as its own category. This would not change the overall revenue of $4,740.00, but it would change the revenue reported for each category. I grouped them together because Apparel fits under the broader merchandise category.